# ⚙️ API Orchestration & Streaming: FastAPI & Server-Sent Events

**My Objective:** I am transitioning the local execution script into a stateless, production-grade API service using FastAPI. Enterprise environments require LLM agents to handle multiple concurrent users, strictly validate incoming payloads, isolate conversational memory per session, and stream outputs to reduce perceived latency.

In [ ]:
import os, sys, shutil, time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "prompting").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "prompting"))
sys.path.insert(0, str(ROOT / "orchestration"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

# Secure, dynamic path resolution
WORK_REPO = Path(os.getenv("WORK_REPO", Path.cwd()))
target = WORK_REPO / "orchestration" / "bankassist_fastAPI" / "app.py"

if target.exists():
    print("Execution Target Found:", target)
else:
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(ROOT / "bankassist_fastAPI" / "app.py", target)
    print("Created Execution Target:", target)

print("\nModel:", os.getenv("MODEL"), "  endpoint:", os.getenv("BASE_URL"))

from fastapi.testclient import TestClient
from check import check_assist_work, load_repo, parse_sse

def show(step):
    """Test Harness CI output formatter."""
    for r in check_assist_work(WORK_REPO):
        if step in r.name or (r.name.startswith(("work repo", "your file")) and not r.passed):
            print(("PASS  " if r.passed else "FAIL  ") + r.name + (f"\n      -> {r.detail}" if r.detail else ""))

def live(fn):
    """Executes live model demonstrations with fallback exception handling."""
    try:
        fn()
    except NotImplementedError as e:
        print("not done yet:", e)
    except Exception as e:
        print(f"live demo skipped ({type(e).__name__}): verify local endpoint or API key.")

## 1. Health Checks & Load Balancing
Infrastructure orchestration requires basic liveness probes. The `/health` endpoint operates without invoking the LLM, allowing Docker containers and load balancers to rapidly ping the service for uptime verification and model configurations.

In [ ]:
show("TEST 1")
mod = load_repo(WORK_REPO)
client = TestClient(mod.app)          

try:
    r = client.get("/health")
    print("\nstatus code:", r.status_code)
    print("body:       ", r.json())
except NotImplementedError as e:
    print("not done yet:", e)

## 2. Input Validation & Cost Control
Unbounded or malformed HTTP payloads cost compute and pose prompt injection risks. By strictly defining `ChatRequest` through Pydantic, FastAPI will automatically reject invalid requests (HTTP 422) *before* any backend LLM execution or token expenditure occurs, enforcing a hard `max_length` of 2000 characters.

In [ ]:
show("TEST 2")
mod = load_repo(WORK_REPO)
from pydantic import ValidationError
for msg in ("What are your Saturday timings?", "", "x" * 5000):
    try:
        mod.ChatRequest(session_id="demo", message=msg)
        print(f"{msg[:30]!r:35} accepted")
    except ValidationError as e:
        print(f"{msg[:30]!r:35} REJECTED: {e.errors()[0]['msg']}")
print("ChatResponse fields:", list(mod.ChatResponse.model_fields))

## 3. Token-Optimized State Management (Sliding Window)
To manage API token limits over long conversations, I implemented a sliding window algorithm. The server prunes historical interactions, maintaining only the most recent user-assistant interactions (`keep_turns`) while continually injecting the static system prompt at the beginning of the context.

In [ ]:
show("TEST 3")
mod = load_repo(WORK_REPO)
from langchain_core.messages import HumanMessage, AIMessage
history = []
for i in range(6):
    history += [HumanMessage(f"q{i}"), AIMessage(f"a{i}")]
try:
    print("history:", " ".join(m.content for m in history))
    print("window :", " ".join(m.content for m in mod.window(history)), f"  (KEEP_TURNS = {mod.KEEP_TURNS})")
except NotImplementedError as e:
    print("not done yet:", e)

## 4. Stateful Execution on a Stateless REST Interface
The API remains stateless; however, the server isolates conversations by generating dynamic memory arrays keyed by a unique `session_id`. This ensures concurrent users cannot leak contextual data into each other's LLM interactions.

In [ ]:
show("TEST 4")
mod = load_repo(WORK_REPO)
client = TestClient(mod.app)
def demo():
    for sid, msg in (("priya", "Hi, my name is Priya and I earn 80,000 a month."),
                     ("priya", "What is my name, and what do I earn?"),
                     ("ravi",  "What is my name?")):
        t = time.time()
        r = client.post("/chat", json={"session_id": sid, "message": msg})
        r.raise_for_status()
        body = r.json()
        print(f"[{sid}] you : {msg}")
        print(f"[{sid}] bank: {body['answer']}")
        print(f"        tools: {body['tool_steps'] or 'none'}   turns: {body['turns']}   {time.time() - t:.1f}s\n")
    print("SESSIONS on the server:", {k: len(v) for k, v in mod.SESSIONS.items()}, "(messages per session)")
live(demo)

## 5. Perceived Latency & Asynchronous Streaming
Synchronous LLM execution creates UI freezing. I refactored the response loop to utilize Server-Sent Events (SSE). The backend yields HTTP chunks asynchronously, drastically reducing time-to-first-token while the execution loop aggregates the text chunks for post-turn memory storage.

In [ ]:
show("TEST 5")
mod = load_repo(WORK_REPO)
client = TestClient(mod.app)
def demo():
    print("--- the raw stream, exactly as a browser receives it (first 500 characters) ---")
    r = client.post("/chat/stream", json={"session_id": "s", "message": "What is the EMI on a 25 lakh home loan at 8.5% for 20 years?"})
    print(r.text[:500])
    events = parse_sse(r.text)
    print("--- events:", len(events), "  kinds:", sorted({k for k, _ in events}))

    print("\n--- why stream? time to the FIRST word vs the WHOLE answer ---")
    from langchain_core.messages import SystemMessage, HumanMessage
    t, first = time.time(), None
    for kind, text in mod.stream_turn(mod.get_llm(), [SystemMessage(mod.SYSTEM), HumanMessage("What documents do I need for a home loan, and how long does approval take?")]):
        if kind == "token" and first is None:
            first = time.time() - t
    print(f"first word after {first:.1f}s, whole answer after {time.time() - t:.1f}s")
live(demo)

## 6. Memory Pruning

In [ ]:
show("TEST 6")

## 7. Edge Cases & Production Reality
In-memory persistence (`SESSIONS` dict) introduces severe limitations at scale. A service restart purges all active conversational contexts, and horizontal scaling behind a load balancer will result in fragmented memories across different server instances. Production-grade environments require shifting this state to an external database or Redis cluster.

In [ ]:
mod = load_repo(WORK_REPO)
client = TestClient(mod.app)
def demo():
    sid = "longchat"
    client.post("/chat", json={"session_id": sid, "message": "My reference number is NW-4471. Please remember it."})
    for q in ("What are your Saturday timings?", "Is there a prepayment penalty on floating loans?",
              "What is the minimum credit score?", "What is the home loan fixed rate?", "Thanks!"):
        client.post("/chat", json={"session_id": sid, "message": q})
    r = client.post("/chat", json={"session_id": sid, "message": "What was my reference number?"}).json()
    print(f"1) after {r['turns'] - 1} turns (window = {mod.KEEP_TURNS}):", r["answer"][:160])

    mod2 = load_repo(WORK_REPO)        # what Ctrl+C + restart does to SESSIONS
    print("2) sessions before restart:", list(mod.SESSIONS), " after restart:", list(mod2.SESSIONS))

    bad = client.post("/chat", json={"session_id": sid, "message": ""})
    print("3) empty message ->", bad.status_code, "- your code never ran, zero tokens spent")
live(demo)

In [ ]:
from prompt_utils import results_table
try:
    print(results_table(check_assist_work(WORK_REPO)))
except Exception as e:
    print(f"Evaluation skipped: {e}")